# Phase 08 - LangGraph 企业级：01 - StateGraph 基础

## 学习目标

1. 理解 StateGraph 的核心概念：State、Node、Edge
2. 掌握 StateGraph 的构建流程：定义State → 添加Node → 添加Edge → 编译
3. 运行第一个 LangGraph 并可视化

## StateGraph 是什么？

StateGraph 是 LangGraph 的核心数据结构，它是一个有向图，其中：
- **State**：图中所有节点共享的类型化状态字典
- **Node**：处理函数（Python函数），接收State，返回State的部分更新
- **Edge**：连接节点的有向边，决定执行顺序

In [ ]:
# 安装依赖（如果尚未安装）
# !pip install langgraph langgraph-checkpoint

from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
import operator

print("✅ LangGraph 导入成功")

## 1. 定义 State

State 是图中所有节点共享的数据结构。使用 TypedDict 定义。

**关键概念：Reducer 函数**

每个 State 字段可以指定一个 **reducer 函数**，用于决定如何处理多个节点对同一字段的更新：
- `operator.add`：将新值追加到列表
- `add_messages`：智能合并消息列表（去重、保留最新）
- 不指定 reducer：直接覆盖

In [ ]:
# 定义一个简单的 State
class SimpleState(TypedDict):
    """
    简单的 State 定义。
    
    messages: 消息列表（使用 add_messages reducer）
    counter: 计数器（覆盖模式）
    greeting: 问候语（覆盖模式）
    """
    messages: Annotated[list, add_messages]
    counter: int
    greeting: str

print("✅ SimpleState 定义完成")
print(f"  字段: messages (list, add_messages reducer)")
print(f"  字段: counter (int)")
print(f"  字段: greeting (str)")

## 2. 创建 Node 函数

Node 函数接收当前 State，返回一个包含 State 部分更新的字典。

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

def node_greet(state: SimpleState) -> dict:
    """节点1：生成问候语。"""
    print(f"  [node_greet] 进入节点，当前 counter={state.get('counter', 0)}")
    new_counter = state.get('counter', 0) + 1
    return {
        "greeting": "Hello from LangGraph!",
        "counter": new_counter,
        "messages": [AIMessage(content=f"你好！我是LangGraph Agent。当前计数: {new_counter}")],
    }


def node_farewell(state: SimpleState) -> dict:
    """节点2：生成告别语。"""
    print(f"  [node_farewell] 进入节点，当前 counter={state.get('counter', 0)}")
    new_counter = state.get('counter', 0) + 1
    return {
        "greeting": f"Goodbye! Visited {new_counter} nodes.",
        "counter": new_counter,
        "messages": [
            AIMessage(content=f"再见！本次执行共处理了 {new_counter} 个节点。")
        ],
    }

print("✅ Node 函数定义完成")
print(f"  node_greet: 生成问候，递增 counter")
print(f"  node_farewell: 生成告别，递增 counter")

## 3. 构建 StateGraph

经典的最小2节点图：START → node_greet → node_farewell → END

In [ ]:
# 创建 StateGraph builder
builder = StateGraph(SimpleState)

# 添加节点
builder.add_node("greet", node_greet)
builder.add_node("farewell", node_farewell)

# 设置入口节点
builder.set_entry_point("greet")

# 添加边（固定路径）
builder.add_edge("greet", "farewell")
builder.add_edge("farewell", END)

# 编译图
graph = builder.compile()

print("✅ StateGraph 构建并编译完成")
print(f"  节点: {list(graph.nodes.keys()) if hasattr(graph, 'nodes') else '见下方可视化'}")
print(f"  入口: greet")
print(f"  路径: greet → farewell → END")

## 4. 调用图（Invoke）

In [ ]:
# 执行图
print("🚀 开始执行图...\n")

result = graph.invoke({
    "counter": 0,
    "greeting": "",
    "messages": [HumanMessage(content="请向我问好")],
})

print(f"\n✅ 图执行完成!")
print(f"\n📊 最终 State:")
print(f"  counter: {result['counter']}")
print(f"  greeting: {result['greeting']}")
print(f"  messages 数: {len(result['messages'])}")
for msg in result['messages']:
    role = "Human" if hasattr(msg, 'content') and msg.__class__.__name__ == 'HumanMessage' else "AI"
    print(f"    [{role}] {msg.content[:80]}")

## 5. 可视化图（Mermaid PNG）

In [ ]:
# 获取图的 Mermaid 表示
print("📊 图的 Mermaid 表示:")
print(graph.get_graph().draw_mermaid())

In [ ]:
# 生成PNG图片（需要安装 graphviz 和 pygraphviz）
# 如果报错，请运行：pip install pygraphviz
# 或使用 Mermaid 在线编辑器：https://mermaid.live/

try:
    from IPython.display import Image, display
    png_data = graph.get_graph().draw_mermaid_png()
    display(Image(png_data))
    print("✅ 图已渲染为 PNG")
except Exception as e:
    print(f"⚠️ 无法生成PNG: {e}")
    print("请使用 draw_mermaid() 获取 Mermaid 代码并在 https://mermaid.live/ 中查看")

## 6. 进阶：带条件路由的图

在实际应用中，图通常包含条件分支。这里展示一个简单的示例。

In [ ]:
import random

class DecisionState(TypedDict):
    """带决策的 State。"""
    query: str
    answer: str
    confidence: float


def node_analyze(state: DecisionState) -> dict:
    """分析节点：评估查询复杂度。"""
    query_len = len(state['query'])
    # 模拟：基于查询长度估算复杂度
    complexity = min(0.95, query_len / 200)
    print(f"  [analyze] 查询长度={query_len}, 估算复杂度={complexity:.2f}")
    return {"confidence": complexity}


def node_simple_answer(state: DecisionState) -> dict:
    """简单回答节点。"""
    print(f"  [simple_answer] 使用简单策略回答")
    return {"answer": f"简单回答: {state['query'][:30]}..."}


def node_complex_answer(state: DecisionState) -> dict:
    """复杂回答节点。"""
    print(f"  [complex_answer] 使用深度分析策略回答")
    return {"answer": f"深度分析回答: {state['query'][:30]}...（经过详细推理）"}


def route_by_confidence(state: DecisionState) -> str:
    """路由函数：根据置信度决定走哪个分支。"""
    if state['confidence'] > 0.5:
        return "complex"
    return "simple"


# 构建图
builder2 = StateGraph(DecisionState)
builder2.add_node("analyze", node_analyze)
builder2.add_node("simple", node_simple_answer)
builder2.add_node("complex", node_complex_answer)

builder2.set_entry_point("analyze")

# 条件边：根据 route_by_confidence 的返回值决定跳转到哪个节点
builder2.add_conditional_edges(
    "analyze",                   # 从 analyze 节点出发
    route_by_confidence,          # 路由函数
    {
        "simple": "simple",       # 如果返回 "simple" → 去 simple 节点
        "complex": "complex",     # 如果返回 "complex" → 去 complex 节点
    }
)

builder2.add_edge("simple", END)
builder2.add_edge("complex", END)

graph2 = builder2.compile()

print("✅ 带条件路由的图已构建")
print(graph2.get_graph().draw_mermaid())

In [ ]:
# 测试短查询（走简单路径）
print("=" * 50)
print("测试短查询:")
print("=" * 50)
result_short = graph2.invoke({"query": "什么是AI?", "answer": "", "confidence": 0.0})
print(f"\n结果: {result_short['answer']}")

print(f"\n{'=' * 50}")
print("测试长查询:")
print("=" * 50)
long_query = "请详细分析人工智能在医疗领域的应用前景，包括诊断辅助、药物发现、个性化治疗和医疗影像分析等多个方面，并结合最新研究数据给出预测。" * 2
result_long = graph2.invoke({"query": long_query, "answer": "", "confidence": 0.0})
print(f"\n结果: {result_long['answer']}")

## 7. 带循环的图（Retry 模式）

LangGraph 支持循环——这是它区别于传统DAG引擎的关键特性。

In [ ]:
class RetryState(TypedDict):
    """带重试的 State。"""
    task: str
    result: str
    attempts: int
    max_attempts: int


def node_process(state: RetryState) -> dict:
    """处理节点。"""
    attempt = state.get('attempts', 0) + 1
    print(f"  [process] 第 {attempt} 次尝试...")
    
    # 模拟：前2次失败，第3次成功
    if attempt < 3:
        return {
            "attempts": attempt,
            "result": f"[失败] 尝试 #{attempt}",
        }
    else:
        return {
            "attempts": attempt,
            "result": f"[成功] 在第 {attempt} 次尝试后完成: {state['task'][:30]}...",
        }


def node_finalize(state: RetryState) -> dict:
    """最终化节点。"""
    print(f"  [finalize] 最终化结果")
    return {"result": f"✅ 已确认: {state['result']}"}


def route_after_process(state: RetryState) -> str:
    """决定是否需要重试。"""
    if "成功" in state.get('result', ''):
        return "finalize"
    if state['attempts'] >= state.get('max_attempts', 3):
        return "finalize"  # 超限也结束
    return "process"  # 重试！回到 process 节点


# 构建循环图
builder3 = StateGraph(RetryState)
builder3.add_node("process", node_process)
builder3.add_node("finalize", node_finalize)

builder3.set_entry_point("process")

# 关键：process → 条件路由 → 可能回到 process（形成循环）
builder3.add_conditional_edges(
    "process",
    route_after_process,
    {
        "process": "process",    # 循环回到自身
        "finalize": "finalize",  # 前进到最终节点
    }
)

builder3.add_edge("finalize", END)

graph3 = builder3.compile()

print("✅ 带循环的图已构建")
print(graph3.get_graph().draw_mermaid())

In [ ]:
# 执行循环图
print("🚀 执行带重试的循环图...\n")

result3 = graph3.invoke({
    "task": "处理复杂的数据分析任务",
    "result": "",
    "attempts": 0,
    "max_attempts": 5,
})

print(f"\n✅ 循环图执行完成")
print(f"  总尝试次数: {result3['attempts']}")
print(f"  最终结果: {result3['result']}")

## 核心要点总结

1. **State**：TypedDict 定义，所有节点共享
2. **Node**：Python 函数，state → partial_state_dict
3. **Edge**：
   - `add_edge(a, b)`：固定路径 a→b
   - `add_conditional_edges(a, fn, mapping)`：条件分支
4. **Compile**：将Builder转为可执行的Graph
5. **Invoke**：传入初始State，获得最终State
6. **循环**：通过条件边将节点指回自身或上游节点

### 下一步

`02-conditional-edges.ipynb` 将深入条件路由的更多高级用法。